# Evo1 Tools Example

This notebook demonstrates how to generate and score DNA sequences using **Evo1**.

## What is Evo1?

[Evo1](https://www.science.org/doi/10.1126/science.ado9336) is a 7B-parameter autoregressive DNA language model trained on prokaryotic and phage genomes, using the StripedHyena architecture for efficient long-context inference.

### Key Features:
- **Generation** -- Autoregressive DNA sequence generation from prompts
- **Scoring** -- Autoregressive likelihood scoring to assess sequence naturalness
- **Multiple checkpoints** -- Base, 131k-context, CRISPR, and transposon fine-tuned variants
- **Flexible sampling** -- Temperature, top-k, and top-p controls for diverse generation

## Imports

## Shared Data Types

### `SequenceScores`
Scoring metrics for a single sequence, returned by the scoring tool. Metrics can be accessed via dict-style (`score.metrics["perplexity"]`) or attribute-style (`score.perplexity`).

| Field | Type | Description |
|-------|------|-------------|
| `metrics` | `Dict[str, float]` | Scoring metrics: `log_likelihood`, `avg_log_likelihood`, `perplexity` |
| `logits` | `Optional[List[List[float]]]` | Per-position logits (seq_len, vocab_size=512). Only present if `return_logits=True` |
| `vocab` | `Optional[List[str]]` | Token ordering for logits columns (512 byte-level tokens) |

In [1]:
from bio_programming_tools.tools.causal_models.evo1 import (
    run_evo1_sample, Evo1SampleInput, Evo1SampleConfig,
    run_evo1_score, Evo1ScoringInput, Evo1ScoringConfig,
)

## 1. Generate DNA Sequences

Generate DNA sequences from a prompt using autoregressive sampling.

Evo1 extends a given DNA prompt by predicting the most likely nucleotides one at a time, using its learned distribution over prokaryotic and phage genomes. The sampling parameters control the diversity and quality of the generated output.

### API Reference

**`Evo1SampleInput`**

| Field | Type | Description |
|-------|------|-------------|
| `prompts` | `List[str]` | Prompt sequences for DNA generation |

**`Evo1SampleConfig`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_name` | `str` | `"evo-1-8k-base"` | Evo1 model checkpoint (`evo-1-8k-base`, `evo-1-131k-base`, `evo-1-8k-crispr`, `evo-1-8k-transposon`) |
| `num_tokens` | `int` | `100` | Number of tokens to generate (does not include prompt) |
| `temperature` | `float` | `1.0` | Scales the randomness of sampling |
| `top_k` | `int` | `4` | Limits sampling to the top-k most probable tokens |
| `top_p` | `float` | `1.0` | Nucleus sampling parameter (cumulative probability cutoff) |
| `prepend_prompt` | `bool` | `False` | Whether to prepend the prompt to the generated sequence |
| `batch_size` | `Optional[int]` | `128` | Max number of prompts per GPU batch |

**`Evo1SampleOutput`**

| Field | Type | Description |
|-------|------|-------------|
| `sequences` | `List[str]` | Generated DNA sequences |
| `scores` | `Optional[List[float]]` | Mean log-probability scores per sequence |

In [2]:
inputs = Evo1SampleInput(prompts=["ACGTACGT"])

config = Evo1SampleConfig(
    model_name="evo-1-8k-base",
    num_tokens=200,
    temperature=0.1,
    top_k=4,
    prepend_prompt=True,
)

sample_result = run_evo1_sample(inputs, config)

prompt = inputs.prompts[0]
generated = sample_result.sequences[0]
print(f"Prompt:       {prompt}")
print(f"Generated:    {generated[:80]}..." if len(generated) > 80 else f"Generated:    {generated}")
print(f"Total length: {len(generated)} nucleotides")
if sample_result.scores:
    print(f"Score:        {sample_result.scores[0]:.4f}")

Prompt:       ACGTACGT
Generated:    ACGTACGTCGCCGAGGAGATCGCCGCCGCGATCGGCGCCGACGTCGTCGTCGAGGGCGTCGGCGGCGCCGACGGCCTCGC...
Total length: 208 nucleotides
Score:        -2.1231


## 2. Score DNA Sequences

Compute autoregressive likelihood metrics for DNA sequences.

Evo1 scores sequences by computing the autoregressive log-likelihood at each position. The resulting perplexity measures how "natural" a DNA sequence looks to the model. Lower values indicate sequences that are more consistent with the patterns learned from natural genomes.

### API Reference

**`Evo1ScoringInput`**

| Field | Type | Description |
|-------|------|-------------|
| `sequences` | `List[str]` | DNA sequences to score |

**`Evo1ScoringConfig`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `model_name` | `str` | `"evo-1-8k-base"` | Evo1 model checkpoint to use |
| `batch_size` | `Optional[int]` | `None` | Max number of sequences on the GPU at once |
| `return_logits` | `bool` | `False` | Whether to include per-position logits in the output |

**`Evo1ScoringOutput`**

| Field | Type | Description |
|-------|------|-------------|
| `scores` | `List[SequenceScores]` | List of scoring outputs, one per input sequence |

In [3]:
score_inputs = Evo1ScoringInput(sequences=["ATGCGTAAA", "ATGCGTAAATTT"])

score_config = Evo1ScoringConfig(batch_size=2, return_logits=False)

score_result = run_evo1_score(score_inputs, score_config)

for i, seq in enumerate(score_inputs.sequences):
    score = score_result.scores[i]
    print(f"Sequence {i + 1}: {seq}")
    print(f"    Log-likelihood:     {score.log_likelihood:.3f}")
    print(f"    Avg log-likelihood: {score.avg_log_likelihood:.3f}")
    print(f"    Perplexity:         {score.perplexity:.3f}")

Sequence 1: ATGCGTAAA
    Log-likelihood:     -13.398
    Avg log-likelihood: -1.489
    Perplexity:         4.431
Sequence 2: ATGCGTAAATTT
    Log-likelihood:     -17.062
    Avg log-likelihood: -1.422
    Perplexity:         4.145


## 3. Export Results

Save the results to files for downstream analysis.

### Supported formats:
- **Sampling**: FASTA, TXT, JSON
- **Scoring**: CSV, JSON

In [4]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export generated sequences to FASTA
sample_result.export(name="evo1_sequences", export_path=output_dir, file_format="fasta")
print(f"Generated sequences exported to {output_dir / 'evo1_sequences.fasta'}")

# Export scores to CSV
score_result.export(name="evo1_scores", export_path=output_dir, file_format="csv")
print(f"Scores exported to {output_dir / 'evo1_scores.csv'}")

Generated sequences exported to example_output/evo1_sequences.fasta
Scores exported to example_output/evo1_scores.csv
